### **Experimental Design Preliminaries**

In [ ]:
# Subjects = what we are experimenting on (people, employees, users, etc.)
# Treatment = some change given to one
# Control Group = the group not given any change

# How to assign subjects to groups? SEE THE DIFFERENCE!
# Option1: non-random ('split' the DataFrame)
group1_nonrandom = DF.iloc[0:100,:]                                              # Example
group2_nonrandom = DF.iloc[100:,:]  
compare_df = pd.concat([group1_nonrandom['var'].describe(), 
                        group2_nonrandom['var'].describe()],axis=1) 
compare_df.columns = ['group1', 'group2']

# Option2: random assignment
group1_random = heights.sample(frac=0.5, replace=False, random_state=42)         # Example 
group2_random = heights.drop(group1_random.index)
compare_df = pd.concat([group1_random['var'].describe(), 
                        group2_random['var'].describe()],axis=1) 
------------------------------------------------------------------------------------------

In [ ]:
# The problem with randomization
# 1) Uneven issue: different number of subjects in groups
# 2) Covariate issue: High variability in some covariates → group imbalances 

# SOLUTION: Block randomization
block1 = DF.sample(frac=0.5, random_state=42, replace=False)     
block1['Block'] = 1 
block2 = DF.drop(block1.index) 
block2['Block'] = 2 
print(len(block1), len(block2))

# Create the 1st block (with two groups Treatment or Control) 
# SOLUTION: Stratified randomization
strata_1 = wealth_data[wealth_data['high_wealth'] == 1] 
strata_1['Block'] = 1
strata_1_g1 = strata_1.sample(100, replace=False)
strata_1_g1['T_C'] = 'T'
strata_1_g2 = strata_1.drop(strata_1_g1.index)
strata_1_g2['T_C'] = 'C'

# Create the 2nd block (with two groups Treatment or Control)
strata_2 = wealth_data[wealth_data['high_wealth'] == 0]
strata_2['Block'] = 2
strata_2_g1 = strata_2.sample(900, replace=False)
strata_2_g1['T_C'] = 'T'
strata_2_g2 = strata_2.drop(strata_2_g1.index)
strata_2_g2['T_C'] = 'C'

wealth_data_stratified = pd.concat([strata_1_g1, strata_1_g2, strata_2_g1, strata_2_g2])
print(wealth_data_stratified.groupby(['Block','T_C', 'high_wealth']).size())
------------------------------------------------------------------------------------------

In [ ]:
# NORMALITY (H0 = "Data is drawn from a Normal Distribution")
from scipy.stats import shapiro                                # 1) Shapiro-Wilk test
alpha = 0.05  
stat, p = shapiro(DF['var']) 
print(f"p: {round(p,4)} test stat: {round(stat,4)}") 

from scipy.stats import anderson                               # 2) Anderson-Darling test
result = anderson(x=DF['var'], dist="norm") 
print(round(result.statistic,4)) 
print(result.significance_level) 
print(result.critical_values) # result Beyond the critical value = Reject H0.

# 3) QQ-Plot

![My Chart](../images/24.png)

### **Experimental Design Techniques**

In [ ]:
# Factorial design: Study multiple independent variables/factors in ONE experiment (Test                                            every combination of factor levels) 
# Discover direct effects and interactions: USING pivot table and heatmap)
------------------------------------------------------------------------------------------
# Randomized Block Design: 
blocks = DF.groupby('var1').apply(lambda x: x.sample(frac=1))   # shuffle within blocks
blocks = blocks.reset_index(drop=True) 

blocks['Treatment'] = np.random.choice(           # random treatment assignment in blocks
    ['ass1', 'ass2', 'ass3'], size=len(blocks))  

sns.boxplot(x='var1', y='var2', hue='Treatment', data=blocks) 

# ANOVA within blocks (f_oneway):
# f_oneway (F-statistics): It tests whether the means of 3 or more independent groups                                                               are significantly different. (H0 = All group means are equal.)
from scipy.stats import f_oneway 
blocks.groupby('Initial_Fitness_Level').apply( 
    lambda x: f_oneway(x[x['Treatment'] == 'ass1']['var2'],  
                       x[x['Treatment'] == 'ass2']['var2'], 
                       x[x['Treatment'] == 'ass3']['var2']))
------------------------------------------------------------------------------------------
# Adjusting for covariates - Further exploring ANCOVA results
# Before anything, we have to merge the experimental and covariate data to create a                                                                      comprehensive dataset, then adeptly performed an ANCOVA.

from statsmodels.formula.api import ols 
model = ols('resp_var ~ exp_var + exp_var1', data=merged_plant_data).fit()  
model.summary()

# NOTE: You can visualize it with sns.lmplot(). Covariates help refine your analysis                                                                 by accounting for additional variability and reducing confounding effects.

### **Analyzing Experimental Data: Statistical Tests and Power**

In [ ]:
# Which test is appropriate?
# "t-test": Comparing means of "2" groups. (Assumptions: normal dist., equal variances)
from scipy.stats import ttest_ind 
group1 = DF[DF['var1'] == 'x_cat']['var2'] 
group2 = DF[DF['var1'] == 'y_cat']['var2'] 
t_stat, p_val = ttest_ind(group1, group2) 
print(f"T-statistic: {t_stat}, P-value: {p_val}")

# "One-way ANOVA": Comparing means across ">2" groups. (Assumptions: equal variances)
from scipy.stats import f_oneway 
var_types = ['cat_x', 'cat_y', 'cat_z'] 
groups = [DF[DF['var1'] == var_types]['var2'] for var in var_types]
f_stat, p_val = f_oneway(*groups) 
print(f"F-statistic: {f_stat}, P-value: {p_val}")

# "Chi-square" (for association): Testing relationships between categorical variables.                                                                                  (No assumptions about distributions)
from scipy.stats import chi2_contingency 
contingency_table = pd.crosstab(DF['var1'], DF['var2'])
chi2_stat, p_val, dof, expected = chi2_contingency(contingency_table) 
print(f"Chi2-statistic: {chi2_stat}, P-value: {p_val}")
------------------------------------------------------------------------------------------
# Post-hoc Tests (To explore pairwise group differences):
# 1. "Tukey's HSD test": Robust for multiple comparisons. AFTER f_oneway ANOVA test above
from statsmodels.stats.multicomp import pairwise_tukeyhsd 
tukey_res = pairwise_tukeyhsd(DF['var1'], DF['var2'], alpha=0.05)
print(tukey_res)

# 2. "Bonferroni Correction": Adjusts p-val to control for Type I errors.
p_values = []                                                         # VERY NICE EXAMPLE
therapy_pairs = [('CBT', 'DBT'), ('CBT', 'ACT'), ('DBT', 'ACT')]
# Conduct t-tests and collect P-values
for pair in therapy_pairs:
    group1 = therapy[therapy['Therapy_Typ'] == pair[0]]['Anxiety_Reduct']
    group2 = therapy[therapy['Therapy_Typ'] == pair[1]]['Anxiety_Reduct']
    t_stat, p_val = ttest_ind(group1, group2)
    p_values.append(p_val)
# Apply Bonferroni correction
print(multipletests(p_values, alpha=0.05, method='bonferroni')[1])
------------------------------------------------------------------------------------------
# Common values: 0.05, 0.01, and 0.10: Reflecting a 5%, 1%, and 10% probability of a TypeI
# Choosing an α Based on the context of the study: Balancing a tolerance for a TypeI error

# Effect size: Quantifies the difference between two groups. It helps us understand how                                                     significant the difference is between the groups, beyond just statistical significance.

# Cohen's d: Standard measure for effect size
def cohens_d(group1, group2): 
    diff = group1.mean() - group2.mean() 
    n1, n2 = len(group1), len(group2) 
    var1, var2 = group1.var(), group2.var() 
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2)) 
    d = diff / pooled_std 
    return d

# Power: the probability of correctly rejecting a false null hypothesis: (1 − β)                                                                    Ranges between 0  and 1 (certainty in ability to detect a true effect)
from statsmodels.stats.power import TTestIndPower 
power_analysis = TTestIndPower()  
power = power_analysis.solve_power(effect_size=1, nobs1=30, alpha=0.05)

#determining the required sample size
required_n = power_analysis.solve_power(effect_size=d, alpha=0.05, power=0.99, ratio=1) 

# NOTE: Trade-off in balancing power and sample size. Larger sample sizes increase power

### **Advanced Insights from Experimental Complexity**

In [ ]:
# Heteroscedasticity: changing variability of a variable across the range of another var.
# Checking Heteroscedasticity
sns.residplot(x='var1', y='var2', data=DF, lowess=True)

# Check normality after/before
from scipy.stats import boxcox 
mineral_rocks['Transformed_var'], _ = boxcox(mineral_rocks['var'])
sns.displot(mineral_rocks['Transformed_var'])
------------------------------------------------------------------------------------------
# Non-parametric tests in experimental designs:
# Plots: 
# 1. Violin plots: Visualizing distributions across groups
condensed_data = DF[DF['var1'].isin(['cat_x', 'cat_y'])] 
sns.violinplot(x='var1', y='var2', data=condensed_data)

# 2. Boxen plot: better insight into distribution shape
sns.boxenplot(x='var1', y='var2', data=DF)

# TESTS: 
# 1. Mann-Whitney U: compare "2" independent groups
from scipy.stats import mannwhitneyu, kruskal 
u_stat, u_pval = mannwhitneyu(DF[DF['var1'] == 'cat_x']['var2'], 
                              DF[Df['var1'] == 'cat_y']['var2']) 
print(f"Mann-Whitney U test p-value: {u_pval:.4f}")

# 2. Kruskal-Wallis: compare more than two groups
k_stat, k_pval = kruskal(DF[DF['var1'] == 'cat_x']['var2'], 
                         DF[Df['var1'] == 'cat_y']['var2'],
                         DF[Df['var1'] == 'cat_z']['var2'])
print(f"Kruskal-Wallis test p-value: {k_pval:.4f}") 